# 05 — k-fqinjët më të afërt (k-NN)

        **Pyetja udhëheqëse:** Si mund të klasifikojmë një gjendje fizike vetëm nga afërsia me shembuj të njohur?

        Ky notebook është ndërtuar si një mësim i plotë kompjuterik. Rruga kryesore ndjek:
        **problem konkret → model matematikor → zbatim nga parimet e para → bibliotekë profesionale → krahasim → interpretim**.

        ## Objektivat
        - Të kuptohet metrika e distancës.
- Të implementohet k-NN nga zero.
- Të analizohet roli i k dhe standardizimit.
- Të ilustrohet mallkimi i dimensionalitetit.

        > **Rruga bazë për orën mësimore (45–60 min)** shënohet me seksionet **CORE**.  
        > Seksionet **EXTENDED** dhe **PLAYGROUND** janë për thellim dhe punë të pavarur.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import time

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("05 — k-fqinjët më të afërt (k-NN)")
print("Seed:", RANDOM_STATE)

# 1. Regjime dinamike si pika në hapësirën e veçorive — CORE

In [ ]:
n=220
reg0=rng.normal([0,0],[0.8,0.7],size=(n,2))
reg1=rng.normal([2.2,1.8],[0.9,0.8],size=(n,2))
X=np.vstack([reg0,reg1]); y=np.r_[np.zeros(n),np.ones(n)]
fig,ax=plt.subplots()
ax.scatter(reg0[:,0],reg0[:,1],s=12,label="regjimi A")
ax.scatter(reg1[:,0],reg1[:,1],s=12,label="regjimi B")
ax.legend(); ax.set_xlabel("amplituda"); ax.set_ylabel("frekuenca relative")
ax.set_title("Dy regjime dinamike")
plt.show()

k-NN nuk ndërton një funksion parametrik. Për një pikë të re $x$, llogarit distancat ndaj të gjitha
mostrave të trajnimit dhe përdor votën e $k$ fqinjëve më të afërt.

# 2. Distanca dhe zbatimi manual — CORE

In [ ]:
def knn_predict(X_train,y_train,X_query,k=5):
    out=[]
    for q in np.asarray(X_query):
        d=np.linalg.norm(X_train-q,axis=1)
        idx=np.argsort(d)[:k]
        labels,counts=np.unique(y_train[idx],return_counts=True)
        out.append(labels[np.argmax(counts)])
    return np.asarray(out)

q=np.array([[1.2,0.8]])
d=np.linalg.norm(X-q,axis=1)
idx=np.argsort(d)[:7]
fig,ax=plt.subplots()
ax.scatter(X[:,0],X[:,1],s=8)
ax.scatter(q[:,0],q[:,1],s=100,marker="x")
ax.scatter(X[idx,0],X[idx,1],s=70,facecolors="none",edgecolors="black")
ax.set_title("7 fqinjët më të afërt të një pike")
plt.show()

### I njëjti algoritëm si modul i ripërdorshëm

In [ ]:
import sys
sys.path.insert(0, str(Path.cwd()))
from src.neighbors.knn import KNNClassifier
knn_src = KNNClassifier(k=7).fit(X, y)
print("Parashikimi src për q:", knn_src.predict(q))

# 3. Kufijtë e vendimit dhe roli i k — CORE

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.3,stratify=y,random_state=RANDOM_STATE)
sc=StandardScaler().fit(Xtr)
Xs,Xts=sc.transform(Xtr),sc.transform(Xte)

def plot_boundary(k):
    xx,yy=np.meshgrid(np.linspace(Xs[:,0].min()-1,Xs[:,0].max()+1,200),
                      np.linspace(Xs[:,1].min()-1,Xs[:,1].max()+1,200))
    pred=knn_predict(Xs,ytr,np.c_[xx.ravel(),yy.ravel()],k).reshape(xx.shape)
    fig,ax=plt.subplots()
    ax.contourf(xx,yy,pred,alpha=0.2)
    ax.scatter(Xs[:,0],Xs[:,1],c=ytr,s=8)
    ax.set_title(f"Kufiri i vendimit, k={k}")
    plt.show()

for k in [1,7,35]:
    plot_boundary(k)

**Interpretim.** k i vogël krijon kufij të ndërlikuar dhe mund të mbipërshtatet; k shumë i madh i zbut kufijtë dhe mund të humbasë strukturë lokale.

# 4. Krahasim me scikit-learn — CORE

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
for k in [1,3,5,9,15,31]:
    m=KNeighborsClassifier(n_neighbors=k).fit(Xs,ytr)
    print(k, accuracy_score(yte,m.predict(Xts)))

# 5. Pse standardizimi është i domosdoshëm — CORE

In [ ]:
from sklearn.pipeline import make_pipeline
X_bad=X.copy()
X_bad[:,1]*=1000  # njësi arbitrare shumë më të mëdha
Xtr,Xte,ytr,yte=train_test_split(X_bad,y,test_size=0.3,stratify=y,random_state=RANDOM_STATE)
raw=KNeighborsClassifier(7).fit(Xtr,ytr).score(Xte,yte)
pipe=make_pipeline(StandardScaler(),KNeighborsClassifier(7)).fit(Xtr,ytr).score(Xte,yte)
raw,pipe

---
## Deri këtu: materiali bazë për orën mësimore

Në një seancë rreth 60 minuta, zakonisht ndalemi këtu. Seksionet që vijojnë janë për thellim,
ushtrime të pavarura dhe eksperimentim.

# 6. Mallkimi i dimensionalitetit — EXTENDED

In [ ]:
dims=np.arange(2,61,4)
ratio=[]
for d in dims:
    pts=rng.normal(size=(700,d))
    q=rng.normal(size=(1,d))
    ds=np.linalg.norm(pts-q,axis=1)
    ratio.append(ds.min()/ds.max())
fig,ax=plt.subplots()
ax.plot(dims,ratio,marker="o")
ax.set_xlabel("Dimensioni"); ax.set_ylabel("d_min / d_max")
ax.set_title("Distancat bëhen më pak dalluese në dimension të lartë")
plt.show()

# 7. Aplikim jo-fizik: njohje shkrimi — EXTENDED

Në klasifikimin e shifrave të shkruara, çdo imazh mund të shihet si një pikë në një hapësirë me qindra dimensione; kjo tregon pse reduktimi dimensional ose metrika më të përshtatshme bëhen të rëndësishme.

# Ushtrime — PLAYGROUND

        ## A. Ushtrime të shkurtra
        1. Krahasoni distancën Euklidiane me Manhattan.
2. Gjeni k optimal me validim të kryqëzuar.
3. Analizoni klasifikimin për një pikë pranë kufirit.
4. Shpjegoni pse k-NN ka kosto të lartë në fazën e parashikimit.

        ## B. Ushtrim i plotë
        Gjeneroni tre regjime dinamike me mbivendosje të ndryshme dhe ndërtoni një hartë të accuracy ndaj k.

        ## C. Playground
        Ndryshoni k, dimensionin, shkallën e veçorive dhe nivelin e zhurmës.

In [ ]:
# TODO / PLAYGROUND
# Ndryshoni parametrat, shtoni grafikë ose krahasoni modele të tjera.